# External Proteomics Data Processing (Figure 6)

This notebook creates deterministic external datasets:

1. `Data/external_data/test_full_proteomics`
2. `Data/external_data/test_significant_proteomics`
3. `Data/external_data/train_full_proteomics_reduced`
4. `Data/external_data/train_significant_proteomics_reduced`

Design:
- Test sets use external proteomics (`full` or `significant-only`).
- Training sets come from `Data/Base` (full proteomics or significant proteomics via `X_proteomics_sig.pkl`) with designated overlapping compounds removed.
- No hardcoded `/mnt/...` paths.


In [4]:
from pathlib import Path
import os
import sys
import shutil
import pickle

import numpy as np
import pandas as pd

# Resolve repo root robustly
cwd = Path.cwd().resolve()
REPO_ROOT = None
for parent in [cwd] + list(cwd.parents):
    if (parent / "Data").exists() and (parent / "notebooks").exists():
        REPO_ROOT = parent
        break
if REPO_ROOT is None:
    candidate = Path('/mnt/jh/nas/PhD/Code/Ms_paint/code_repository')
    REPO_ROOT = candidate if candidate.exists() else cwd

os.chdir(REPO_ROOT)

for src in [REPO_ROOT / "notebooks", REPO_ROOT / "scripts"]:
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

from data_processing.proteomics import prepare_proteomics_data

print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /mnt/nas/PhD/Code/Ms_paint/code_repository


In [5]:
# ---- Paths / constants ----
FASTA_PATH = Path("Data/Fasta_files/Ecoli.fasta")

RAW_MS_FILES = {
    "kanamycin":     Path("Data/external_data/MS_files/251126_kanamycin_vs_ctrl.txt"),
    "imipenem":      Path("Data/external_data/MS_files/251126_imipenem_vs_ctrl.txt"),
    "cefotaxime":    Path("Data/external_data/MS_files/251126_cefotaxime_vs_ctrl.txt"),
    "ciprofloxacin": Path("Data/external_data/MS_files/251126_ciprofloxacin_vs_ctrl.txt"),
}

# Keep a clean folder with only the 4 intended raw files for robust parsing
FULL_MS_DIR = Path("Data/external_data/MS_files/raw_selected")
FULL_MS_DIR.mkdir(parents=True, exist_ok=True)

SIG_MS_DIR = Path("Data/external_data/MS_files/statsFiltered_combined")
SIG_MS_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR = Path("Data/Base")
SIG_PROT_PATH = Path("Data/Base/X_proteomics_sig.pkl")

OUT_COMPOUND = Path("Data/external_data/Compound_data")
OUT_COMPOUND_SIG = OUT_COMPOUND / "significant"
OUT_COMPOUND.mkdir(parents=True, exist_ok=True)
OUT_COMPOUND_SIG.mkdir(parents=True, exist_ok=True)

OUT_TEST_FULL = Path("Data/external_data/test_full_proteomics")
OUT_TEST_SIG = Path("Data/external_data/test_significant_proteomics")
OUT_TRAIN_FULL = Path("Data/external_data/train_full_proteomics_reduced")
OUT_TRAIN_SIG = Path("Data/external_data/train_significant_proteomics_reduced")
for p in [OUT_TEST_FULL, OUT_TEST_SIG, OUT_TRAIN_FULL, OUT_TRAIN_SIG]:
    p.mkdir(parents=True, exist_ok=True)

# Label dict must match prepare_proteomics_data() index construction: JH<sampleprefix>_<rep>
EXTERNAL_LABEL_DICT = {
    "JHkanamycin": "Protein synthesis",
    "JHimipenem": "Cell wall biosynthesis",
    "JHciprofloxacin": "DNA replication",
    "JHcefotaxime": "Cell wall biosynthesis",
}

# Rename from JH-prefixed parsed IDs to publication-readable IDs
ID_PREFIX_MAP = {
    "JHkanamycin": "Kanamycin",
    "JHimipenem": "Imipenem",
    "JHciprofloxacin": "Ciprofloxacin",
    "JHcefotaxime": "Cefotaxime",
}

# Remove potentially overlapping compounds from base training (same as prior analyses)
BASE_EXCLUSION_PREFIXES = {"JH29", "JH55", "JH6"}

PVAL_THRESHOLD = 1.3  # -log10(p) threshold (~p<0.05)


In [6]:
# ---- Prepare clean raw-selected folder ----
for ab, src in RAW_MS_FILES.items():
    if not src.exists():
        raise FileNotFoundError(f"Missing raw MS file: {src}")
    dst = FULL_MS_DIR / src.name
    shutil.copy2(src, dst)

print("raw_selected files:")
for p in sorted(FULL_MS_DIR.glob("*.txt")):
    print(" -", p)


raw_selected files:
 - Data/external_data/MS_files/raw_selected/251126_cefotaxime_vs_ctrl.txt
 - Data/external_data/MS_files/raw_selected/251126_ciprofloxacin_vs_ctrl.txt
 - Data/external_data/MS_files/raw_selected/251126_imipenem_vs_ctrl.txt
 - Data/external_data/MS_files/raw_selected/251126_kanamycin_vs_ctrl.txt


In [7]:
# ---- Build significant-filtered combined external MS file ----
def process_antibiotic_file(path: Path, ab_name: str, pval_threshold: float = 1.3) -> pd.DataFrame:
    df = pd.read_csv(path, sep="	")

    rep_cols = [c for c in df.columns if c.startswith(f"{ab_name}_")]
    if not rep_cols:
        raise ValueError(f"No replicate columns found for {ab_name} in {path}")

    pval_cols = [c for c in df.columns if "N: -Log Student's T-test p-value" in c]
    if len(pval_cols) != 1:
        raise ValueError(f"Expected 1 p-value column in {path}, found {pval_cols}")
    pval_col = pval_cols[0]

    sig_mask = (df[pval_col] > pval_threshold).fillna(False)
    df.loc[~sig_mask, rep_cols] = 0

    meta_cols = [c for c in df.columns if c.startswith("T: ")]
    key_col = "T: Protein.Ids"
    if key_col in meta_cols:
        meta_cols = [key_col] + [c for c in meta_cols if c != key_col]

    return df[rep_cols + meta_cols].copy()

processed = {ab: process_antibiotic_file(path, ab, PVAL_THRESHOLD) for ab, path in RAW_MS_FILES.items()}

key_col = "T: Protein.Ids"
for ab, df_ab in processed.items():
    if key_col not in df_ab.columns:
        raise KeyError(f"{key_col} missing in processed frame for {ab}")

ab_order = ["kanamycin", "imipenem", "cefotaxime", "ciprofloxacin"]
combined = processed[ab_order[0]].copy()
for ab in ab_order[1:]:
    rep_cols = [c for c in processed[ab].columns if not c.startswith("T: ")]
    combined = combined.merge(processed[ab][[key_col] + rep_cols], on=key_col, how="outer")

intensity_cols = []
for ab in ab_order:
    cols = [c for c in combined.columns if c.startswith(f"{ab}_")]
    cols = sorted(cols, key=lambda x: int(x.split("_")[-1]))
    intensity_cols.extend(cols)

meta_cols = [c for c in combined.columns if c.startswith("T: ")]
if key_col in meta_cols:
    meta_cols = [key_col] + [c for c in meta_cols if c != key_col]

combined = combined[intensity_cols + meta_cols].copy()

sig_file = SIG_MS_DIR / "External_statsFiltered.txt"
combined.to_csv(sig_file, sep="	", index=False)
print("Saved:", sig_file)
print("shape:", combined.shape)


Saved: Data/external_data/MS_files/statsFiltered_combined/External_statsFiltered.txt
shape: (2190, 20)


In [8]:
# ---- Build external proteomics matrices (full + significant-only) ----
X_prot_full, _, _ = prepare_proteomics_data(
    str(FASTA_PATH), str(FULL_MS_DIR), EXTERNAL_LABEL_DICT, exclude_labels=[]
)
X_prot_sig, _, _ = prepare_proteomics_data(
    str(FASTA_PATH), str(SIG_MS_DIR), EXTERNAL_LABEL_DICT, exclude_labels=[]
)

assert X_prot_full.shape[1] == 4401, f"Expected 4401 features (full), got {X_prot_full.shape[1]}"
assert X_prot_sig.shape[1] == 4401, f"Expected 4401 features (sig), got {X_prot_sig.shape[1]}"
assert len(X_prot_full) > 0 and len(X_prot_sig) > 0, "Empty proteomics matrix generated"

# Rename sample IDs for compatibility with figure notebooks

def prettify_ids(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = out.index.astype(str)
    new_idx = []
    for sid in out.index:
        pref = sid.split("_")[0]
        rest = sid.split("_", 1)[1] if "_" in sid else "1"
        new_pref = ID_PREFIX_MAP.get(pref, pref)
        new_idx.append(f"{new_pref}_{rest}")
    out.index = pd.Index(new_idx)
    return out

X_prot_full = prettify_ids(X_prot_full)
X_prot_sig = prettify_ids(X_prot_sig)

# Build y_labels from final sample IDs
label_by_compound = {
    "Kanamycin": "Protein synthesis",
    "Imipenem": "Cell wall biosynthesis",
    "Ciprofloxacin": "DNA replication",
    "Cefotaxime": "Cell wall biosynthesis",
}

def build_labels(df: pd.DataFrame) -> pd.Series:
    idx = df.index.astype(str)
    y = pd.Series(index=idx, dtype=object, name="Label")
    for sid in idx:
        cmpd = sid.split("_")[0]
        y.loc[sid] = label_by_compound.get(cmpd, "Unknown")
    return y

y_full = build_labels(X_prot_full)
y_sig = build_labels(X_prot_sig)
assert (y_full != "Unknown").all() and (y_sig != "Unknown").all(), "Unknown labels present"

X_prot_full.to_pickle(OUT_COMPOUND / "X_proteomics.pkl")
y_full.to_pickle(OUT_COMPOUND / "y_labels.pkl")

X_prot_sig.to_pickle(OUT_COMPOUND_SIG / "X_proteomics.pkl")
y_sig.to_pickle(OUT_COMPOUND_SIG / "y_labels.pkl")

print("Saved full compound data:", OUT_COMPOUND)
print("Saved sig compound data:", OUT_COMPOUND_SIG)
print("full IDs:", X_prot_full.index.tolist())


[Warning] 34703 missing values filled with 0 in combined fingerprint matrix.


/mnt/nas/PhD/Code/Ms_paint/code_repository/notebooks/data_processing/proteomics.py:165: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(',', '.', regex=True)


[Warning] 26227 missing values filled with 0 in combined fingerprint matrix.
Saved full compound data: Data/external_data/Compound_data
Saved sig compound data: Data/external_data/Compound_data/significant
full IDs: ['Kanamycin_1', 'Kanamycin_2', 'Kanamycin_3', 'Imipenem_1', 'Imipenem_2', 'Imipenem_3', 'Cefotaxime_1', 'Cefotaxime_2', 'Cefotaxime_3', 'Ciprofloxacin_1', 'Ciprofloxacin_2', 'Ciprofloxacin_3']


In [9]:
# ---- Save external test sets ----
common_ids = sorted(set(X_prot_full.index.astype(str)).intersection(set(X_prot_sig.index.astype(str))))
assert len(common_ids) == 12, f"Expected 12 external samples, got {len(common_ids)}"

X_prot_full.loc[common_ids].to_pickle(OUT_TEST_FULL / "X_proteomics.pkl")
y_full.loc[common_ids].to_pickle(OUT_TEST_FULL / "y_labels.pkl")

X_prot_sig.loc[common_ids].to_pickle(OUT_TEST_SIG / "X_proteomics.pkl")
y_sig.loc[common_ids].to_pickle(OUT_TEST_SIG / "y_labels.pkl")

print("Saved:", OUT_TEST_FULL)
print("Saved:", OUT_TEST_SIG)


Saved: Data/external_data/test_full_proteomics
Saved: Data/external_data/test_significant_proteomics


In [10]:
# ---- Build reduced base training sets (full + significant proteomics) ----
def load_base_dataset(base_dir: Path, proteomics_override: Path | None = None):
    prot_path = proteomics_override if proteomics_override is not None else (base_dir / "X_proteomics.pkl")
    with open(prot_path, "rb") as f: Xp = pickle.load(f)
    with open(base_dir / "X_ecfp.pkl", "rb") as f: Xe = pickle.load(f)
    with open(base_dir / "X_cellpaint.pkl", "rb") as f: Xc = pickle.load(f)
    with open(base_dir / "X_mic.pkl", "rb") as f: Xm = pickle.load(f)
    with open(base_dir / "X_gc.pkl", "rb") as f: Xg = pickle.load(f)
    with open(base_dir / "y_labels.pkl", "rb") as f: y = pickle.load(f)

    if isinstance(y, dict):
        y = pd.Series(y)
    elif isinstance(y, pd.DataFrame):
        if "Label" in y.columns:
            y = y["Label"]
        elif y.shape[1] == 1:
            y = y.iloc[:, 0]
        else:
            raise ValueError(f"Unexpected y_labels columns: {list(y.columns)}")

    for d in [Xp, Xe, Xc, Xm, Xg]:
        d.index = d.index.astype(str)
    y.index = y.index.astype(str)

    return {"X_proteomics": Xp, "X_ecfp": Xe, "X_cellpaint": Xc, "X_mic": Xm, "X_gc": Xg, "y_labels": y}

full_src = load_base_dataset(BASE_DIR)
sig_src = load_base_dataset(BASE_DIR, proteomics_override=SIG_PROT_PATH)

def save_reduced_train(out_dir: Path, src: dict):
    keep_ids = [sid for sid in src["X_proteomics"].index if sid.split("_")[0] not in BASE_EXCLUSION_PREFIXES]
    src["X_proteomics"].loc[keep_ids].to_pickle(out_dir / "X_proteomics.pkl")
    src["X_ecfp"].loc[keep_ids].to_pickle(out_dir / "X_ecfp.pkl")
    src["X_cellpaint"].loc[keep_ids].to_pickle(out_dir / "X_cellpaint.pkl")
    src["X_mic"].loc[keep_ids].to_pickle(out_dir / "X_mic.pkl")
    src["X_gc"].loc[keep_ids].to_pickle(out_dir / "X_gc.pkl")
    src["y_labels"].loc[keep_ids].to_pickle(out_dir / "y_labels.pkl")
    return len(keep_ids)

n_full = save_reduced_train(OUT_TRAIN_FULL, full_src)
n_sig = save_reduced_train(OUT_TRAIN_SIG, sig_src)

print("Saved:", OUT_TRAIN_FULL, "rows:", n_full)
print("Saved:", OUT_TRAIN_SIG, "rows:", n_sig)


Saved: Data/external_data/train_full_proteomics_reduced rows: 188
Saved: Data/external_data/train_significant_proteomics_reduced rows: 188


In [11]:
# ---- Validation ----
def load_test(path: Path):
    with open(path / "X_proteomics.pkl", "rb") as f: Xp = pickle.load(f)
    with open(path / "y_labels.pkl", "rb") as f: y = pickle.load(f)
    if isinstance(y, dict): y = pd.Series(y)
    elif isinstance(y, pd.DataFrame):
        if "Label" in y.columns: y = y["Label"]
        elif y.shape[1] == 1: y = y.iloc[:, 0]
    Xp.index = Xp.index.astype(str)
    y.index = y.index.astype(str)
    return Xp, y

def load_train(path: Path):
    with open(path / "X_proteomics.pkl", "rb") as f: Xp = pickle.load(f)
    with open(path / "X_ecfp.pkl", "rb") as f: Xe = pickle.load(f)
    with open(path / "X_cellpaint.pkl", "rb") as f: Xc = pickle.load(f)
    with open(path / "X_mic.pkl", "rb") as f: Xm = pickle.load(f)
    with open(path / "X_gc.pkl", "rb") as f: Xg = pickle.load(f)
    with open(path / "y_labels.pkl", "rb") as f: y = pickle.load(f)
    if isinstance(y, dict): y = pd.Series(y)
    elif isinstance(y, pd.DataFrame):
        if "Label" in y.columns: y = y["Label"]
        elif y.shape[1] == 1: y = y.iloc[:, 0]
    for d in [Xp, Xe, Xc, Xm, Xg]: d.index = d.index.astype(str)
    y.index = y.index.astype(str)
    return Xp, Xe, Xc, Xm, Xg, y

Xt_full, yt_full = load_test(OUT_TEST_FULL)
Xt_sig, yt_sig = load_test(OUT_TEST_SIG)

assert Xt_full.index.equals(yt_full.index)
assert Xt_sig.index.equals(yt_sig.index)
assert len(Xt_full) == len(Xt_sig) == 12

Tr_full = load_train(OUT_TRAIN_FULL)
Tr_sig = load_train(OUT_TRAIN_SIG)

for name, ds in [("train_full", Tr_full), ("train_sig", Tr_sig)]:
    Xp, Xe, Xc, Xm, Xg, y = ds
    idx = Xp.index
    assert idx.equals(Xe.index) and idx.equals(Xc.index) and idx.equals(Xm.index) and idx.equals(Xg.index) and idx.equals(y.index), f"Index mismatch in {name}"
    assert not any(i.split("_")[0] in BASE_EXCLUSION_PREFIXES for i in idx), f"Excluded prefixes still present in {name}"

print("Validation passed")
print("test_full rows:", len(Xt_full), "test_sig rows:", len(Xt_sig))
print("train_full rows:", len(Tr_full[0]), "train_sig rows:", len(Tr_sig[0]))


Validation passed
test_full rows: 12 test_sig rows: 12
train_full rows: 188 train_sig rows: 188
